# Hugging Face Pipelines: Practical Inference Showcase

A reproducible, notebook-based walkthrough of common inference workflows with
[Hugging Face Transformers](https://huggingface.co/docs/transformers) and
[Diffusers](https://huggingface.co/docs/diffusers).

This project demonstrates sentiment analysis, named entity recognition,
question answering, summarization, translation, zero-shot classification,
text generation, image generation, and text-to-speech.

> **Scope:** This is an inference project. It uses pretrained models to generate
> predictions; it does not train or fine-tune models.

## Project overview

Hugging Face **pipelines** provide a high-level interface for common machine
learning tasks. A pipeline packages model loading and task-specific
preprocessing/postprocessing behind a callable Python object.

The basic workflow is:

1. Select a task and (optionally) a model.
2. Create the pipeline.
3. Pass input data to the pipeline and inspect the result.

```python
analyzer = pipeline("sentiment-analysis")
analyzer("The results are impressive.")
```

### Learning objectives

- Understand the difference between model training and inference.
- Select and configure pipelines for different tasks.
- Run inference on CPU or an available accelerator.
- Compare default task models with explicitly selected models.
- Understand practical considerations such as model downloads, memory, and authentication.

## Environment setup

### Requirements

- Python 3.10 or newer
- Jupyter Notebook or JupyterLab
- Internet access for downloading pretrained models
- Optional: NVIDIA GPU with a compatible CUDA-enabled PyTorch installation
- Optional: Hugging Face access token for gated models or higher download limits

Install the project dependencies from the repository root:

```bash
python -m pip install -r requirements.txt
```

**PyTorch note:** Install the PyTorch build appropriate for your operating system
and accelerator from [pytorch.org](https://pytorch.org/get-started/locally/).
The requirements file intentionally does not force a CUDA-specific PyTorch build.

For Google Colab, open this notebook and run the setup cell below. A GPU is
recommended for image generation and larger models, but most text examples can
run on CPU (more slowly).

In [ ]:
import platform
import torch
from transformers import pipeline

def get_pipeline_device():
    """Choose an available accelerator, otherwise fall back to CPU."""
    if torch.cuda.is_available():
        return 0  # First CUDA GPU
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"  # Apple Silicon
    return -1  # CPU

DEVICE = get_pipeline_device()
DEVICE_NAME = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else ("Apple Metal (MPS)" if DEVICE == "mps" else "CPU")
)

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"Selected device: {DEVICE_NAME}")

## Optional: Hugging Face authentication

Most examples in this notebook use public models and do not require login.
Authentication may be needed for gated/private models or to increase access
limits.

Set a token as an environment variable named `HF_TOKEN`, or add a Colab secret
with that name. Never commit tokens, passwords, or other credentials to GitHub.

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN")

# In Google Colab, optionally read the token from Colab Secrets.
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except (ImportError, Exception):
    pass

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Authenticated with Hugging Face.")
else:
    print("No HF_TOKEN found. Continuing with public models.")

## 1. Sentiment analysis

Sentiment analysis classifies the emotional tone of text. The default pipeline
selects a task-appropriate pretrained model automatically.

In [ ]:
sentiment_analyzer = pipeline("sentiment-analysis", device=DEVICE)

examples = [
    "I'm excited to be learning about practical AI engineering.",
    "I expected this project to be easier, and I'm disappointed.",
]

for text in examples:
    print({"text": text, "prediction": sentiment_analyzer(text)})

### Use a specific sentiment model

Specifying a model makes the model choice explicit. The example below uses a
multilingual model that predicts a star rating rather than only positive or
negative sentiment.

In [ ]:
multilingual_sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=DEVICE,
)

print(multilingual_sentiment(
    "The Hugging Face pipeline API is simple and useful!"
))

## 2. Named entity recognition

Named entity recognition (NER) identifies entities such as people,
organizations, and locations in text.

In [ ]:
ner = pipeline("ner", aggregation_strategy="simple", device=DEVICE)

text = (
    "AI engineers are exploring Hugging Face pipelines "
    "with colleagues in Google Colab."
)

for entity in ner(text):
    print(entity)

## 3. Question answering

A question-answering pipeline extracts an answer from a supplied context.
The answer is grounded in the context passed to the model.

In [ ]:
question = "What are Hugging Face pipelines?"
context = (
    "Pipelines are a high-level API for running pretrained models "
    "on common machine learning tasks."
)

question_answerer = pipeline("question-answering", device=DEVICE)
answer = question_answerer(question=question, context=context)
print(answer)

## 4. Text summarization

Summarization produces a shorter version of a longer passage. Output quality
and length depend on the model and the input.

In [ ]:
summarizer = pipeline("summarization", device=DEVICE)

article = """
The Hugging Face Transformers library provides tools for working with
pretrained transformer models. It supports tasks such as text classification,
named entity recognition, question answering, translation, and summarization.
Its pipeline API offers a convenient interface for common inference workflows.
"""

summary = summarizer(
    article,
    max_length=60,
    min_length=20,
    do_sample=False,
)
print(summary[0]["summary_text"])

## 5. Translation

The first example uses the task's default model. The second selects a specific
English-to-Spanish model.

In [ ]:
english_to_french = pipeline("translation_en_to_fr", device=DEVICE)
french_result = english_to_french(
    "The team was impressed by the simplicity of the pipeline API."
)
print(french_result[0]["translation_text"])

In [ ]:
english_to_spanish = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-es",
    device=DEVICE,
)
spanish_result = english_to_spanish(
    "The team was impressed by the simplicity of the pipeline API."
)
print(spanish_result[0]["translation_text"])

## 6. Zero-shot classification

Zero-shot classification assigns labels supplied at runtime. The candidate
labels do not need to be part of a task-specific training dataset.

In [ ]:
classifier = pipeline("zero-shot-classification", device=DEVICE)

classification = classifier(
    "Hugging Face makes pretrained AI models easier to use.",
    candidate_labels=["technology", "sports", "politics"],
)
print(classification)

## 7. Text generation

A text-generation pipeline continues a prompt using a pretrained language
model. Generated text is probabilistic and should be reviewed before use.

In [ ]:
generator = pipeline("text-generation", device=DEVICE)

generated = generator(
    "A practical benefit of using pretrained machine learning pipelines is",
    max_new_tokens=60,
    do_sample=True,
    temperature=0.7,
    return_full_text=True,
)
print(generated[0]["generated_text"])

## 8. Image generation with Diffusers

Diffusers supports diffusion-based image generation. This example uses
**SDXL Turbo**, which is designed for a small number of inference steps.

Image generation can require substantial memory and is best run with a
compatible GPU. The cell deliberately checks for CUDA rather than failing
silently on a CPU-only machine. Review the model card and its license before
using generated images in a commercial or public project.

In [ ]:
if not torch.cuda.is_available():
    print("Image generation skipped: a CUDA-enabled GPU is recommended.")
else:
    from diffusers import AutoPipelineForText2Image

    image_pipe = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/sdxl-turbo",
        torch_dtype=torch.float16,
        variant="fp16",
    ).to("cuda")

    prompt = "A vibrant editorial illustration of engineers building AI systems"
    generated_image = image_pipe(
        prompt=prompt,
        num_inference_steps=4,
        guidance_scale=0.0,
    ).images[0]

    display(generated_image)

## 9. Text-to-speech with SpeechT5

This section converts text into speech. SpeechT5 uses a speaker embedding;
the example obtains one from the CMU Arctic x-vector dataset.

Dataset access and model loading may take time. The dataset is loaded from
Hugging Face and is not bundled with this repository.

In [ ]:
if not torch.cuda.is_available():
    print("Text-to-speech skipped in this demo: a CUDA-enabled GPU is recommended.")
else:
    from datasets import load_dataset
    from IPython.display import Audio, display

    tts = pipeline(
        "text-to-speech",
        model="microsoft/speecht5_tts",
        device=DEVICE,
    )

    speaker_dataset = load_dataset(
        "Matthijs/cmu-arctic-xvectors",
        split="validation",
    )
    speaker_embedding = torch.tensor(
        speaker_dataset[7306]["xvector"]
    ).unsqueeze(0)

    speech = tts(
        "Hello! This is a demonstration of text to speech.",
        forward_params={"speaker_embeddings": speaker_embedding},
    )

    display(Audio(speech["audio"], rate=speech["sampling_rate"]))

## Key takeaways

- A pipeline is a callable interface for a specific inference task.
- You can let Transformers choose a default model or specify one explicitly.
- Device selection affects performance and hardware compatibility.
- Model downloads can be large; the first run is often slower than later runs.
- Outputs from generative models should be checked for accuracy, quality, and suitability.

## Further reading

- [Transformers pipelines documentation](https://huggingface.co/docs/transformers/main_classes/pipelines)
- [Transformers task summary](https://huggingface.co/docs/transformers/tasks)
- [Diffusers pipeline overview](https://huggingface.co/docs/diffusers/en/api/pipelines/overview)
- [Hugging Face model hub](https://huggingface.co/models)
- [PyTorch installation guide](https://pytorch.org/get-started/locally/)

---

*Project note: This repository is an educational demonstration. Model behavior,
availability, hardware requirements, and licenses are determined by the
individual pretrained models and their publishers.*